# NLP Analysis on Book Descriptions

In [ ]:
import pandas as pd
import nltk
import spacy
from nltk import pos_tag
from nltk.tokenize import word_tokenize
from nltk.chunk import RegexpParser

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

nlp = spacy.load("en_core_web_sm")

In [ ]:
books = pd.read_csv("books_preprocessed.csv")
books = books.dropna(subset=["description"])
print(f"Total books: {len(books)}")

sample_text = books["description"].iloc[0]
print(f"\nSample book: {books['title'].iloc[0]}")
print(f"Description: {sample_text[:300]}...")

## 1. POS Tagging

In [ ]:
tokens = word_tokenize(sample_text)
pos_tags = pos_tag(tokens)

print("POS Tags (first 20 tokens):")
for word, tag in pos_tags[:20]:
    print(f"  {word:20s} \u2192 {tag}")

print("\nPOS Tag Distribution:")
tag_counts = {}
for _, tag in pos_tags:
    tag_counts[tag] = tag_counts.get(tag, 0) + 1
for tag, count in sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"  {tag}: {count}")

In [ ]:
doc = nlp(sample_text[:500])

print("spaCy POS Tags:")
for token in doc[:20]:
    print(f"  {token.text:20s} \u2192 {token.pos_:10s} ({token.tag_})")

## 2. Chunking (Noun Phrase Extraction)

In [ ]:
grammar = r"""
    NP: {<DT|PP\$>?<JJ>*<NN.*>+}
    VP: {<VB.*><NP|PP>}
"""
chunk_parser = RegexpParser(grammar)

tokens = word_tokenize(sample_text[:500])
tagged = pos_tag(tokens)
tree = chunk_parser.parse(tagged)

print("Noun Phrases found:")
noun_phrases = []
for subtree in tree.subtrees(filter=lambda t: t.label() == 'NP'):
    np_text = " ".join(word for word, tag in subtree.leaves())
    noun_phrases.append(np_text)
    print(f"  \u2022 {np_text}")

print(f"\nTotal noun phrases: {len(noun_phrases)}")

print("\nParse Tree (first few branches):")
print(tree)

## 3. Morphological Analysis

In [ ]:
doc = nlp(sample_text[:500])

print("Morphological Analysis:")
print(f"{'Token':20s} {'Lemma':15s} {'POS':8s} {'Morph Features'}")
print("-" * 80)
for token in doc[:25]:
    print(f"  {token.text:20s} {token.lemma_:15s} {token.pos_:8s} {token.morph}")

print("\n\nPrefix/Suffix Analysis:")
sample_words = ["unbelievable", "storytelling", "preacher", "redemption", "imagination", "beautifully"]
for word in sample_words:
    doc_word = nlp(word)
    token = doc_word[0]
    print(f"  {word}")
    print(f"    Lemma: {token.lemma_}")
    print(f"    POS: {token.pos_}")
    print(f"    Morphology: {token.morph}")
    if word.startswith("un") or word.startswith("re") or word.startswith("im"):
        prefix = word[:2] if word[:2] in ["un", "re", "im"] else ""
        print(f"    Prefix: {prefix}-")
    if word.endswith("ing") or word.endswith("tion") or word.endswith("ly") or word.endswith("er") or word.endswith("able"):
        for suffix in ["able", "tion", "ing", "ly", "er"]:
            if word.endswith(suffix):
                print(f"    Suffix: -{suffix}")
                break
    print()

## 4. Named Entity Recognition (NER)

In [ ]:
doc = nlp(sample_text)

print(f"Named Entities in '{books['title'].iloc[0]}':")
print(f"{'Entity':30s} {'Label':15s} {'Description'}")
print("-" * 70)
for ent in doc.ents:
    print(f"  {ent.text:30s} {ent.label_:15s} {spacy.explain(ent.label_)}")

In [ ]:
from collections import Counter

all_persons = []
all_locations = []
all_orgs = []

for idx, row in books.iterrows():
    desc = str(row["description"])[:1000]
    doc = nlp(desc)
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            all_persons.append(ent.text)
        elif ent.label_ in ["GPE", "LOC"]:
            all_locations.append(ent.text)
        elif ent.label_ == "ORG":
            all_orgs.append(ent.text)

print("Top 15 Persons mentioned:")
for name, count in Counter(all_persons).most_common(15):
    print(f"  {name}: {count}")

print("\nTop 15 Locations mentioned:")
for loc, count in Counter(all_locations).most_common(15):
    print(f"  {loc}: {count}")

print("\nTop 15 Organizations mentioned:")
for org, count in Counter(all_orgs).most_common(15):
    print(f"  {org}: {count}")

## 5. Extract & Save NER Data

In [ ]:
def extract_entities(text):
    if not isinstance(text, str):
        return "", "", ""
    doc = nlp(text[:1000])
    persons = [ent.text for ent in doc.ents if ent.label_ == "PERSON"]
    locations = [ent.text for ent in doc.ents if ent.label_ in ["GPE", "LOC"]]
    orgs = [ent.text for ent in doc.ents if ent.label_ == "ORG"]
    return ";".join(persons), ";".join(locations), ";".join(orgs)

print("Extracting entities from all books...")
entity_data = books["description"].apply(extract_entities)
books["persons"] = [e[0] for e in entity_data]
books["locations"] = [e[1] for e in entity_data]
books["organizations"] = [e[2] for e in entity_data]

books.to_csv("books_with_entities.csv", index=False)
print(f"Saved books_with_entities.csv with NER columns")
print(f"\nBooks with location data: {(books['locations'] != '').sum()}")
print(f"Books with person data: {(books['persons'] != '').sum()}")